In [1]:
import pandas as pd
import os

In [2]:
raw_path = os.path.join('..', 'data', 'raw')
processed_path = os.path.join('..', 'data', 'processed')

## Estrutura demográfica e Pressão Urbana

### Densidade Populacional 2021

In [3]:
pop_density_file = os.path.join(raw_path, 'densidade_populacional_freguesia_2011_2021.csv')

try:
    # Reading with semicolon delimiter (standard for Portuguese INE/Census exports)
    densidade_populacional = pd.read_csv(pop_density_file, delimiter=";")

    # 3. Professional Renaming (English for Thesis/Babel)
    densidade_populacional = densidade_populacional.rename(columns={
        "Local de residência à data dos Censos [2021] (NUTS - 2013)": "Freguesia",
        "S7A2011:2011-T:HM": "Densidade_Populacional_2011",
        "S7A2021:2021-T:HM": "Densidade_Populacional_2021"
    })

    # 4. Cleaning the Parish string (Removing the code prefix '011231: Parish Name')
    densidade_populacional['Freguesia'] = densidade_populacional['Freguesia'].str.split(':', n=1).str[1].str.strip()

    print(f"✅ Population Density data loaded: {densidade_populacional.shape[0]} parishes identified.")

except FileNotFoundError:
    print(f"❌ Error: File not found at {pop_density_file}")
    raise

✅ Population Density data loaded: 24 parishes identified.


In [4]:
# apagar a coluna de 2011 para densidade populacional
densidade_populacional_2021 = densidade_populacional.drop('Densidade_Populacional_2011', axis=1)

In [5]:
densidade_populacional_2021.head()

,Freguesia,Densidade_Populacional_2021
0,Ajuda,"4967,36"
1,Alcântara,"2731,76"
2,Alvalade,"6237,64"
3,Areeiro,"12302,33"
4,Arroios,"15634,74"


### Indice de Envelhecimento 2021

In [6]:
indice_envelhecimento_path = os.path.join(raw_path, 'indice_envelhecimento_2011_2021.xls')

In [7]:
indice_envehecimento = pd.read_excel(indice_envelhecimento_path)
indice_envehecimento = indice_envehecimento.iloc[10:, :]
# renomear a primiera coluna para freguesia
indice_envehecimento.rename(columns={indice_envehecimento.columns[0]: 'Freguesia'}, inplace=True)
## remover a coluna a seguir a freguesia
indice_envehecimento = indice_envehecimento.drop(indice_envehecimento.columns[1], axis=1)
## remover a ultima coluna
indice_envehecimento = indice_envehecimento.drop(indice_envehecimento.columns[-1], axis=1)
indice_envehecimento = indice_envehecimento.drop(indice_envehecimento.columns[2], axis=1)
## remove lines with nan values
indice_envehecimento = indice_envehecimento.dropna()
## rename columns
indice_envehecimento.columns = ['Freguesia', 'Indice_Envelhecimento_2011', 'Indice_Envelhecimento_2021']
# drop column of 2011
indice_envehecimento_2021 = indice_envehecimento.drop('Indice_Envelhecimento_2011', axis=1)

In [8]:
indice_envehecimento_2021

,Freguesia,Indice_Envelhecimento_2021
10,Ajuda,250.54
11,Alcântara,245.4
12,Alvalade,239.34
13,Areeiro,208.12
14,Arroios,236.25
15,Avenidas Novas,209.78
16,Beato,216.59
17,Belém,183.08
18,Benfica,263.21
19,Campo de Ourique,221.51


### Proporção População Estrangeira 2021

In [9]:
proporcao_populacao_residente_estrangeira_path = os.path.join(raw_path, 'proporcao_populacao_residente_estrangeira_2011_2021.csv')
proporcao_populacao_residente_estrangeira = pd.read_csv(proporcao_populacao_residente_estrangeira_path, delimiter=";")

In [10]:
proporcao_populacao_residente_estrangeira.columns = ['Freguesia', 'Proporcao_Populacao_Residente_Estrangeira_2011', 'Proporcao_Populacao_Residente_Estrangeira_2021']
proporcao_populacao_residente_estrangeira['Freguesia'] = proporcao_populacao_residente_estrangeira['Freguesia'].str.split(':', n=1).str[1]
# remove column from 2011
proporcao_populacao_residente_estrangeira_2021 = proporcao_populacao_residente_estrangeira.drop('Proporcao_Populacao_Residente_Estrangeira_2011', axis=1)

In [11]:
# proporcao_populacao_residente_estrangeira_2021 : object to float (replace comma with period)
proporcao_populacao_residente_estrangeira_2021['Proporcao_Populacao_Residente_Estrangeira_2021'] = proporcao_populacao_residente_estrangeira_2021['Proporcao_Populacao_Residente_Estrangeira_2021'].str.replace(',', '.').astype(float)

In [12]:
proporcao_populacao_residente_estrangeira_2021

,Freguesia,Proporcao_Populacao_Residente_Estrangeira_2021
0,Ajuda,7.28
1,Alcântara,11.75
2,Alvalade,5.38
3,Areeiro,7.86
4,Arroios,23.33
5,Avenidas Novas,9.97
6,Beato,11.84
7,Belém,7.12
8,Benfica,5.98
9,Campo de Ourique,11.26


In [13]:
censos = pd.merge(indice_envehecimento_2021, proporcao_populacao_residente_estrangeira_2021, on='Freguesia', how='left')

In [14]:
censos = pd.merge(censos, densidade_populacional_2021, on='Freguesia', how='left')

## Dinâmica Populacional

### Variação População Residente (2011-2021)

In [15]:
populacao_residente_path = os.path.join(raw_path, 'populacao_residente_2011_2021.csv')
populacao_residente = pd.read_csv(populacao_residente_path, delimiter=";")

In [16]:
## rename columns
populacao_residente = populacao_residente.rename(columns={
    "Local de residência à data dos Censos [2021] (NUTS - 2013)": "Freguesia",
    "S7A2011:2011-T:HM-T:Total": "Populacao_Residente_2011",
    "S7A2021:2021-T:HM-T:Total": "Populacao_Residente_2021"

})
populacao_residente['Freguesia'] = populacao_residente['Freguesia'].str.split(':', n=1).str[1]

In [17]:
populacao_residente

,Freguesia,Populacao_Residente_2011,Populacao_Residente_2021
0,Ajuda,15617,14306
1,Alcântara,13943,13850
2,Alvalade,31813,33309
3,Areeiro,20131,21160
4,Arroios,31653,33302
5,Avenidas Novas,21625,23261
6,Beato,12737,12183
7,Belém,16528,16546
8,Benfica,36985,35362
9,Campo de Ourique,22120,22140


In [18]:
## variação da população residente entre 2011 e 2021
populacao_residente_2021 = populacao_residente.drop('Populacao_Residente_2011', axis=1)
populacao_residente_2021 = populacao_residente_2021.drop('Populacao_Residente_2021', axis=1)
populacao_residente_2021['Variacao_Relativa_Populacao_Residente_2011_2021'] = ((populacao_residente['Populacao_Residente_2021'] - populacao_residente['Populacao_Residente_2011']) / populacao_residente['Populacao_Residente_2011'])* 100

Variação Relativa / Taxa de Crescimento (Qual a intensidade em %?)

In [19]:
populacao_residente_2021

,Freguesia,Variacao_Relativa_Populacao_Residente_2011_2021
0,Ajuda,-8.394698
1,Alcântara,-0.667001
2,Alvalade,4.702480
3,Areeiro,5.111520
4,Arroios,5.209617
5,Avenidas Novas,7.565318
6,Beato,-4.349533
7,Belém,0.108906
8,Benfica,-4.388266
9,Campo de Ourique,0.090416


In [20]:
censos = pd.merge(censos, populacao_residente_2021, on='Freguesia', how='left')

In [21]:
censos

,Freguesia,Indice_Envelhecimento_2021,Proporcao_Populacao_Residente_Estrangeira_2021,Densidade_Populacional_2021,Variacao_Relativa_Populacao_Residente_2011_2021
0,Ajuda,250.54,7.28,"4967,36",-8.394698
1,Alcântara,245.4,11.75,"2731,76",-0.667001
2,Alvalade,239.34,5.38,"6237,64",4.702480
3,Areeiro,208.12,7.86,"12302,33",5.111520
4,Arroios,236.25,23.33,"15634,74",5.209617
5,Avenidas Novas,209.78,9.97,"7779,6",7.565318
6,Beato,216.59,11.84,"4912,5",-4.349533
7,Belém,183.08,7.12,"1586,39",0.108906
8,Benfica,263.21,5.98,"4409,23",-4.388266
9,Campo de Ourique,221.51,11.26,"13418,18",0.090416


## Estrutura Habitacional

### Número de Alojamentos Clássicos 2021

In [22]:
alojamentos_path = os.path.join(raw_path, 'alojamentos_familiares_classicos_formaocupacao_2011_2021.csv')
alojamentos_familiares_classicos = pd.read_csv(alojamentos_path, delimiter=";")

In [23]:
## rename columns
alojamentos_familiares_classicos = alojamentos_familiares_classicos.rename(columns={
    "Localização geográfica à data dos Censos [2021] (NUTS - 2013)": "Freguesia",
    "S7A2011:2011-1:Residência habitual-T:Total-T:Total": "Número_Residencia_Habitual_2011",
    "S7A2011:2011-2:Residência secundária-T:Total-T:Total": "Número_Residencia_Secundaria_2011",
    "S7A2011:2011-3:Vago para venda ou arrendamento-T:Total-T:Total": "Número_Vago_Venda_Arrendamento_2011",
    "S7A2011:2011-4:Vago por outros motivos-T:Total-T:Total": "Número_Vago_Outros_Motivos_2011",
    "S7A2021:2021-1:Residência habitual-T:Total-T:Total": "Número_Residencia_Habitual_2021",
    "S7A2021:2021-2:Residência secundária-T:Total-T:Total": "Número_Residencia_Secundaria_2021",
    "S7A2021:2021-3:Vago para venda ou arrendamento-T:Total-T:Total": "Número_Vago_Venda_Arrendamento_2021",
    "S7A2021:2021-4:Vago por outros motivos-T:Total-T:Total": "Número_Vago_Outros_Motivos_2021"
})
alojamentos_familiares_classicos['Freguesia'] = alojamentos_familiares_classicos['Freguesia'].str.split(':', n=1).str[1]

## make a total for 2021
alojamentos_familiares_classicos['Número_Total_Alojamentos_2021'] = alojamentos_familiares_classicos['Número_Residencia_Habitual_2021'] + alojamentos_familiares_classicos['Número_Residencia_Secundaria_2021'] + alojamentos_familiares_classicos['Número_Vago_Venda_Arrendamento_2021'] + alojamentos_familiares_classicos['Número_Vago_Outros_Motivos_2021']

### Densidade Alojamentos 2021

In [24]:
# Desnidade de alojamentos familiares clássicos em 2021
areas_freguesias = {
    'Ajuda': 2.88,
    'Alcântara': 5.07,
    'Alvalade': 5.34,
    'Areeiro': 1.74,
    'Arroios': 2.13,
    'Avenidas Novas': 2.99,
    'Beato': 2.48,
    'Belém': 10.43,
    'Benfica': 8.02,
    'Campo de Ourique': 1.65,
    'Campolide': 2.77,
    'Carnide': 3.69,
    'Estrela': 4.60,
    'Lumiar': 6.57,
    'Marvila': 7.12,
    'Misericórdia': 2.19,
    'Olivais': 8.09,
    'Parque das Nações': 5.44,
    'Penha de França': 2.71,
    'Santa Clara': 3.36,
    'Santa Maria Maior': 3.01,
    'Santo António': 1.49,
    'São Domingos de Benfica': 4.29,
    'São Vicente': 1.99
}
alojamentos_familiares_classicos['Area_Freguesia_km2'] = alojamentos_familiares_classicos['Freguesia'].map(areas_freguesias)
alojamentos_familiares_classicos['Densidade_Alojamentos_Familiares_Classicos_2021'] = alojamentos_familiares_classicos['Número_Total_Alojamentos_2021'] / alojamentos_familiares_classicos['Area_Freguesia_km2']

### Percentagem Residencias Habituais 2021

In [25]:
### percentagem de alojamentos familiares clássicos que são residência habitual em 2021
alojamentos_familiares_classicos['Percentagem_Residencia_Habitual_2021'] = (alojamentos_familiares_classicos['Número_Residencia_Habitual_2021'] / alojamentos_familiares_classicos['Número_Total_Alojamentos_2021']) * 100

### Percentagem Residencias Secundárias 2021

In [26]:
### percentagem de alojamentos familiares clássicos que são residência secundária em 2021
alojamentos_familiares_classicos['Percentagem_Residencia_Secundaria_2021'] = (alojamentos_familiares_classicos['Número_Residencia_Secundaria_2021'] / alojamentos_familiares_classicos['Número_Total_Alojamentos_2021']) * 100

## Pressão Habitacional

In [27]:
agregados_domesticos_privados_path = os.path.join(raw_path, 'numero_agregados_domesticos_privados_e_dimensao_2011_2021.csv')
agregados_domesticos_privados = pd.read_csv(agregados_domesticos_privados_path, delimiter=";")

In [28]:

## rename columns
agregados_domesticos_privados_2021 = agregados_domesticos_privados.rename(columns={
    "Local de residência à data dos Censos [2021] (NUTS - 2013)": "Freguesia",
    "S7A2021:2021-01:Com 1 pessoa-T:Total": "Número_Agregados_1_Pessoa_2021",
    "S7A2021:2021-02:Com 2 pessoas-T:Total": "Número_Agregados_2_Pessoas_2021",
    "S7A2021:2021-03:Com 3 pessoas-T:Total": "Número_Agregados_3_Pessoas_2021",
    "S7A2021:2021-04:Com 4 pessoas-T:Total": "Número_Agregados_4_Pessoas_2021",
    "S7A2021:2021-05:Com 5 pessoas-T:Total": "Número_Agregados_5_Pessoas_2021",
    "S7A2021:2021-06:Com 6 pessoas-T:Total": "Número_Agregados_6_Pessoas_2021",
    "S7A2021:2021-07:Com 7 pessoas-T:Total": "Número_Agregados_7_Pessoas_2021",
    "S7A2021:2021-08:Com 8 pessoas-T:Total": "Número_Agregados_8_Pessoas_2021",
    "S7A2021:2021-09:Com 9 ou mais pessoas-T:Total": "Número_Agregados_9_Mais_Pessoas_2021"
})
agregados_domesticos_privados_2021['Freguesia'] = agregados_domesticos_privados_2021['Freguesia'].str.split(':', n=1).str[1]

In [29]:
agregados_domesticos_privados_2021

,Freguesia,Número_Agregados_1_Pessoa_2021,Número_Agregados_2_Pessoas_2021,Número_Agregados_3_Pessoas_2021,Número_Agregados_4_Pessoas_2021,Número_Agregados_5_Pessoas_2021,Número_Agregados_6_Pessoas_2021,Número_Agregados_7_Pessoas_2021,Número_Agregados_8_Pessoas_2021,Número_Agregados_9_Mais_Pessoas_2021
0,Ajuda,2379,2160,1098,605,202,72,33,2,9
1,Alcântara,2655,2182,1014,559,181,56,16,6,4
2,Alvalade,5328,4656,2370,1734,515,113,39,13,12
3,Areeiro,3362,3046,1484,1068,349,101,28,7,13
4,Arroios,5702,4936,2177,1388,479,169,92,42,69
5,Avenidas Novas,3937,3286,1512,1109,432,119,34,9,7
6,Beato,2101,1962,905,506,161,52,19,6,5
7,Belém,2445,2283,1081,872,340,97,26,6,5
8,Benfica,5949,5767,2561,1607,395,123,52,23,21
9,Campo de Ourique,4094,3290,1427,1067,355,103,27,6,2


### Indice de Pressão Habitacional 2021
Medir a pressão real sobre a habitação disponível para quem vive lá, tens de mudar o denominador:$$\frac{\text{Agregados Domésticos Privados}}{\text{Alojamentos de Residência Habitual}}$$Isto diz se as casas que estão a ser usadas estão sobrelotadas (Índice > 1) ou se há gente a viver sozinha (Índice < 1).

Se o resultado for 1,05: "Há 1,05 famílias por cada casa ocupada". Ou seja, tens partilha de casa (coabitação). Se for 0,95, tens casas ocupadas sem agregados "clássicos" (ex: estudantes deslocados não censitários, ou erros de estatística), ou simplesmente folga.

In [30]:
## indice de pressão habitacional em 2021
agregados_domesticos_privados_2021["Total_Agregados_2021"] = agregados_domesticos_privados_2021['Número_Agregados_1_Pessoa_2021'] + agregados_domesticos_privados_2021['Número_Agregados_2_Pessoas_2021'] + agregados_domesticos_privados_2021['Número_Agregados_3_Pessoas_2021'] + agregados_domesticos_privados_2021['Número_Agregados_4_Pessoas_2021'] + agregados_domesticos_privados_2021['Número_Agregados_5_Pessoas_2021'] + agregados_domesticos_privados_2021['Número_Agregados_6_Pessoas_2021'] + agregados_domesticos_privados_2021['Número_Agregados_7_Pessoas_2021'] + agregados_domesticos_privados_2021['Número_Agregados_8_Pessoas_2021'] + agregados_domesticos_privados_2021['Número_Agregados_9_Mais_Pessoas_2021']

In [31]:
## indice de pressão habitacional em 2021
alojamentos_familiares_classicos['Indice_Pressao_Habitacional_2021'] = agregados_domesticos_privados_2021['Total_Agregados_2021'] / alojamentos_familiares_classicos['Número_Residencia_Habitual_2021']

### Percentagem Fogos Vagos

In [32]:
alojamentos_familiares_classicos['Pct_Fogos_Vagos_2021'] = (alojamentos_familiares_classicos['Número_Vago_Venda_Arrendamento_2021'] + alojamentos_familiares_classicos['Número_Vago_Outros_Motivos_2021']) / alojamentos_familiares_classicos['Número_Total_Alojamentos_2021'] * 100


In [33]:
alojamentos_familiares_classicos

,Freguesia,Número_Residencia_Habitual_2011,Número_Residencia_Secundaria_2011,Número_Vago_Venda_Arrendamento_2011,Número_Vago_Outros_Motivos_2011,Número_Residencia_Habitual_2021,Número_Residencia_Secundaria_2021,Número_Vago_Venda_Arrendamento_2021,Número_Vago_Outros_Motivos_2021,Número_Total_Alojamentos_2021,Area_Freguesia_km2,Densidade_Alojamentos_Familiares_Classicos_2021,Percentagem_Residencia_Habitual_2021,Percentagem_Residencia_Secundaria_2021,Indice_Pressao_Habitacional_2021,Pct_Fogos_Vagos_2021
0,Ajuda,6912,1094,384,493,6549,949,604,709,8811,2.88,3059.375000,74.327545,10.770628,1.001680,14.901827
1,Alcântara,6512,944,753,677,6668,682,601,907,8858,5.07,1747.140039,75.276586,7.699255,1.000750,17.024159
2,Alvalade,14104,2477,642,1534,14765,1659,888,1559,18871,5.34,3533.895131,78.241747,8.791267,1.001016,12.966986
3,Areeiro,8991,1553,782,1203,9452,1319,1028,730,12529,1.74,7200.574713,75.440977,10.527576,1.000635,14.031447
4,Arroios,14327,2582,1544,2558,15025,1933,1559,2331,20848,2.13,9787.793427,72.069263,9.271873,1.001930,18.658864
5,Avenidas Novas,9607,2005,969,1846,10432,2087,1034,1224,14777,2.99,4942.140468,70.596197,14.123300,1.001246,15.280503
6,Beato,5672,794,303,1015,5713,507,399,900,7519,2.48,3031.854839,75.980849,6.742918,1.000700,17.276234
7,Belém,6997,913,551,978,7154,791,550,879,9374,10.43,898.753595,76.317474,8.438233,1.000140,15.244293
8,Benfica,16577,2223,807,1682,16488,1958,998,1436,20880,8.02,2603.491272,78.965517,9.377395,1.000607,11.657088
9,Campo de Ourique,10346,1432,777,1222,10370,998,978,1278,13624,1.65,8256.969697,76.115678,7.325308,1.000096,16.559014


### Estrutura dos Agregados

"Para o cálculo da dimensão média, na categoria aberta '9 ou mais indivíduos', foi considerado o valor do limite inferior (9). Esta opção metodológica, embora introduza um ligeiro viés negativo na estimativa, garante que não há sobreavaliação da dimensão dos agregados baseada em pressupostos arbitrários sobre a cauda da distribuição."

"Foi efetuada uma análise de sensibilidade para o valor atribuído à categoria '9 ou mais' (testando os valores 9 e 12). Verificou-se que, devido à reduzida representatividade destes agregados (<0.5% do total), a variação na média final é estatisticamente insignificante (<0.01), não alterando as conclusões. Manteve-se o valor conservador de 9."

In [34]:
# 1. Calcular a "Massa Populacional" fixa (de 1 a 8 pessoas)
# Isto evita repetires esta soma gigante 4 vezes.
populacao_fixa_1_a_8 = (
    (1 * agregados_domesticos_privados_2021['Número_Agregados_1_Pessoa_2021']) +
    (2 * agregados_domesticos_privados_2021['Número_Agregados_2_Pessoas_2021']) +
    (3 * agregados_domesticos_privados_2021['Número_Agregados_3_Pessoas_2021']) +
    (4 * agregados_domesticos_privados_2021['Número_Agregados_4_Pessoas_2021']) +
    (5 * agregados_domesticos_privados_2021['Número_Agregados_5_Pessoas_2021']) +
    (6 * agregados_domesticos_privados_2021['Número_Agregados_6_Pessoas_2021']) +
    (7 * agregados_domesticos_privados_2021['Número_Agregados_7_Pessoas_2021']) +
    (8 * agregados_domesticos_privados_2021['Número_Agregados_8_Pessoas_2021'])
)

# 2. Criar um DataFrame temporário para veres a comparação
comparacao = pd.DataFrame()

# Se tiveres uma coluna com o nome da Freguesia, adiciona aqui para facilitar a leitura:
if 'Freguesia' in agregados_domesticos_privados_2021.columns:
    comparacao['Freguesia'] = agregados_domesticos_privados_2021['Freguesia']

# 3. Loop de Teste (Cenários 9, 10, 11, 12)
pesos_teste = [9, 10, 11, 12]

for peso in pesos_teste:
    # População Estimada = Parte Fixa + (Agregados >9 * Peso Testado)
    pop_estimada = populacao_fixa_1_a_8 + (peso * agregados_domesticos_privados_2021['Número_Agregados_9_Mais_Pessoas_2021'])
    
    # Calcular a média
    col_name = f'Media_Peso_{peso}'
    comparacao[col_name] = pop_estimada / agregados_domesticos_privados_2021['Total_Agregados_2021']

# 4. Calcular o impacto máximo (Diferença entre o cenário 12 e o 9)
comparacao['Diferenca_Impacto'] = comparacao['Media_Peso_12'] - comparacao['Media_Peso_9']

# 5. Ver os resultados (arredondados a 4 casas decimais)
print("--- ANÁLISE DE SENSIBILIDADE ---")
print(comparacao.round(4).head(10)) # Mostra as primeiras 10 linhas

print(f"\nO impacto máximo registado foi de: {comparacao['Diferenca_Impacto'].max():.5f}")

--- ANÁLISE DE SENSIBILIDADE ---
          Freguesia  Media_Peso_9  Media_Peso_10  Media_Peso_11  \
0             Ajuda        2.1620         2.1634         2.1648   
1         Alcântara        2.0581         2.0587         2.0593   
2          Alvalade        2.1938         2.1946         2.1954   
3           Areeiro        2.2096         2.2109         2.2123   
4           Arroios        2.1700         2.1746         2.1792   
5    Avenidas Novas        2.1760         2.1766         2.1773   
6             Beato        2.1177         2.1186         2.1195   
7             Belém        2.2780         2.2787         2.2794   
8           Benfica        2.1241         2.1254         2.1267   
9  Campo de Ourique        2.1089         2.1091         2.1092   

   Media_Peso_12  Diferenca_Impacto  
0         2.1662             0.0041  
1         2.0599             0.0018  
2         2.1962             0.0024  
3         2.2137             0.0041  
4         2.1837             0.0138  
5

Realizou-se uma análise de sensibilidade variando o peso atribuído à categoria aberta '9 ou mais' entre 9 e 12 indivíduos. Verificou-se que o impacto máximo desta escolha metodológica na dimensão média dos agregados foi de 0.03, um valor estatisticamente negligenciável que não altera a ordenação das freguesias nem as conclusões do estudo. Consequentemente, optou-se pelo limiar inferior (9).

In [35]:
# --- DECISÃO FINAL ---
agregados_domesticos_privados_2021['Dimensao_Media_Agregados_2021'] = comparacao['Media_Peso_9']

In [36]:
censos = pd.merge(censos, alojamentos_familiares_classicos, on='Freguesia', how='left')
censos = pd.merge(censos, agregados_domesticos_privados_2021, on='Freguesia', how='left')

In [37]:
censos

,Freguesia,Indice_Envelhecimento_2021,Proporcao_Populacao_Residente_Estrangeira_2021,Densidade_Populacional_2021,Variacao_Relativa_Populacao_Residente_2011_2021,Número_Residencia_Habitual_2011,Número_Residencia_Secundaria_2011,Número_Vago_Venda_Arrendamento_2011,Número_Vago_Outros_Motivos_2011,Número_Residencia_Habitual_2021,...,Número_Agregados_2_Pessoas_2021,Número_Agregados_3_Pessoas_2021,Número_Agregados_4_Pessoas_2021,Número_Agregados_5_Pessoas_2021,Número_Agregados_6_Pessoas_2021,Número_Agregados_7_Pessoas_2021,Número_Agregados_8_Pessoas_2021,Número_Agregados_9_Mais_Pessoas_2021,Total_Agregados_2021,Dimensao_Media_Agregados_2021
0,Ajuda,250.54,7.28,"4967,36",-8.394698,6912,1094,384,493,6549,...,2160,1098,605,202,72,33,2,9,6560,2.162043
1,Alcântara,245.4,11.75,"2731,76",-0.667001,6512,944,753,677,6668,...,2182,1014,559,181,56,16,6,4,6673,2.058145
2,Alvalade,239.34,5.38,"6237,64",4.702480,14104,2477,642,1534,14765,...,4656,2370,1734,515,113,39,13,12,14780,2.193775
3,Areeiro,208.12,7.86,"12302,33",5.111520,8991,1553,782,1203,9452,...,3046,1484,1068,349,101,28,7,13,9458,2.209558
4,Arroios,236.25,23.33,"15634,74",5.209617,14327,2582,1544,2558,15025,...,4936,2177,1388,479,169,92,42,69,15054,2.169988
5,Avenidas Novas,209.78,9.97,"7779,6",7.565318,9607,2005,969,1846,10432,...,3286,1512,1109,432,119,34,9,7,10445,2.175969
6,Beato,216.59,11.84,"4912,5",-4.349533,5672,794,303,1015,5713,...,1962,905,506,161,52,19,6,5,5717,2.117719
7,Belém,183.08,7.12,"1586,39",0.108906,6997,913,551,978,7154,...,2283,1081,872,340,97,26,6,5,7155,2.277987
8,Benfica,263.21,5.98,"4409,23",-4.388266,16577,2223,807,1682,16488,...,5767,2561,1607,395,123,52,23,21,16498,2.124136
9,Campo de Ourique,221.51,11.26,"13418,18",0.090416,10346,1432,777,1222,10370,...,3290,1427,1067,355,103,27,6,2,10371,2.108861


## Capital Humano / Estrutura Socioeconómica

### Proporção da População residente com Ensino Superior Completo 2021

In [38]:
proporcao_populacao_residente_com_ensinosuperior = pd.read_csv(os.path.join(raw_path, 'proporcao_populacao_residente_com_ensinosuperior_2011_2021.csv'), delimiter=";")

In [39]:
proporcao_populacao_residente_com_ensinosuperior.columns = ['Freguesia', 'Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011', 'Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021']
proporcao_populacao_residente_com_ensinosuperior['Freguesia'] = proporcao_populacao_residente_com_ensinosuperior['Freguesia'].str.split(':', n=1).str[1]
proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011'] = proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011'].str.replace(',', '.').astype(float)
proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021'] = proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021'].str.replace(',', '.').astype(float)

In [40]:
proporcao_populacao_residente_com_ensinosuperior_2021 = proporcao_populacao_residente_com_ensinosuperior.drop('Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011', axis=1)

In [41]:
proporcao_populacao_residente_com_ensinosuperior_2021

,Freguesia,Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021
0,Ajuda,28.30
1,Alcântara,40.31
2,Alvalade,56.03
3,Areeiro,55.88
4,Arroios,46.49
5,Avenidas Novas,61.13
6,Beato,26.08
7,Belém,56.95
8,Benfica,40.47
9,Campo de Ourique,47.01


### Variação Proporção da População Residente com Ensino Superior Completo (2011-2021)

In [42]:
## variação proporcao de população residente com ensino superior completo entre 2011 e 2021
proporcao_populacao_residente_com_ensinosuperior_2021['Variacao_absoluta_Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011_2021'] = proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021'] - proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011']
proporcao_populacao_residente_com_ensinosuperior_2021

,Freguesia,Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021,Variacao_absoluta_Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011_2021
0,Ajuda,28.30,11.85
1,Alcântara,40.31,13.37
2,Alvalade,56.03,13.04
3,Areeiro,55.88,11.77
4,Arroios,46.49,14.56
5,Avenidas Novas,61.13,11.83
6,Beato,26.08,8.57
7,Belém,56.95,9.67
8,Benfica,40.47,10.45
9,Campo de Ourique,47.01,12.86


In [101]:
## variação proporcao de população residente com ensino superior completo entre 2011 e 2021
proporcao_populacao_residente_com_ensinosuperior_2021['Variacao_relativa_Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011_2021'] = ((proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021'] - proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011']) / proporcao_populacao_residente_com_ensinosuperior['Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011']) * 100
proporcao_populacao_residente_com_ensinosuperior_2021

,Freguesia,Proporcao_Populacao_Residente_Ensino_Superior_Completo_2021,Variacao_absoluta_Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011_2021,Variacao_relativa_Proporcao_Populacao_Residente_Ensino_Superior_Completo_2011_2021
0,Ajuda,28.30,11.85,72.036474
1,Alcântara,40.31,13.37,49.628805
2,Alvalade,56.03,13.04,30.332635
3,Areeiro,55.88,11.77,26.683292
4,Arroios,46.49,14.56,45.599749
5,Avenidas Novas,61.13,11.83,23.995943
6,Beato,26.08,8.57,48.943461
7,Belém,56.95,9.67,20.452623
8,Benfica,40.47,10.45,34.810127
9,Campo de Ourique,47.01,12.86,37.657394


In [43]:
censos = pd.merge(censos, proporcao_populacao_residente_com_ensinosuperior_2021, on='Freguesia', how='left')

## Edificado

### Percentagem Edificos Antigos, Intermediários e Novos

In [44]:
df_censos = pd.read_excel(os.path.join(raw_path, 'Censos2021_Edificios.xls'), header=7)

In [45]:
# remover a primiera coluna vazia
df_censos = df_censos.iloc[:, 1:]
# renomear a primiera coluna para freguesia
df_censos.rename(columns={df_censos.columns[0]: 'Freguesia'}, inplace=True)
## remover a coluna a seguir a freguesia
df_censos = df_censos.drop(df_censos.columns[1], axis=1)
# Remover linhas vazias e espaços extra
df_censos = df_censos.dropna(subset=['Freguesia'])
df_censos['Freguesia'] = df_censos['Freguesia'].astype(str).str.strip()
## remover colunas com missing values
df_censos = df_censos.dropna(axis=1, how='all')

# 3. Definir quais são as colunas de Época (As que vão virar valores na nova coluna)
# Copiei exatamente os nomes da tua imagem
colunas_epoca = [
    'Antes de 1919', '1919 - 1945', '1946 - 1960', '1961 - 1980', 
    '1981 - 1990', '1991 - 2000', '2001 - 2005', '2006 - 2010', 
    '2011 - 2015', '2016 - 2021'
]

colunas_epoca = [
    'Antes de 1919', '1919 - 1945', '1946 - 1960', '1961 - 1980', 
    '1981 - 1990', '1991 - 2000', '2001 - 2005', '2006 - 2010', 
    '2011 - 2015', '2016 - 2021'
]

# Garantir que são números (tratamento de erros do INE)
for col in colunas_epoca:
    df_censos[col] = pd.to_numeric(df_censos[col], errors='coerce').fillna(0)

In [46]:
# Definição dos grupos (Copia exatamente como aparece no Excel do INE)
cols_antigo = ['Antes de 1919', '1919 - 1945', '1946 - 1960']
cols_intermediario = [ '1961 - 1980', '1981 - 1990', '1991 - 2000']
cols_moderno = ['2001 - 2005', '2006 - 2010', '2011 - 2015', '2016 - 2021']
# Colunas intermédias (apenas para calcular o total correto)
cols_todas = cols_antigo + cols_intermediario + cols_moderno

# Calcular Total (Denominador)
df_censos['Total_Edificios'] = df_censos[cols_todas].sum(axis=1)

# Variavel Antigo
df_censos['PCT_Edificios_Antigos'] = df_censos[cols_antigo].sum(axis=1) / df_censos['Total_Edificios']
# Variavel Expansão
df_censos['PCT_Edificios_Intermediarios'] = df_censos[cols_intermediario].sum(axis=1) / df_censos['Total_Edificios']
# Variavel Moderno
df_censos['PCT_Edificios_Modernos'] = df_censos[cols_moderno].sum(axis=1) / df_censos['Total_Edificios']

df_final_censos = df_censos[['Freguesia', 'PCT_Edificios_Antigos', 'PCT_Edificios_Intermediarios', 'PCT_Edificios_Modernos']]

# df_final_censos.to_csv("edificios_censos_estruturais.csv", index=False, encoding='utf-8-sig')

In [47]:
censos = pd.merge(censos, df_final_censos, on='Freguesia', how='left')

In [48]:
output_dir = os.path.join('..', 'data', 'processed')

# Garantir que a pasta existe
if not os.path.exists(output_dir):
    os.makedirs(output_dir)


output_filename = 'CENSOS_dados.csv'
output_path = os.path.join(output_dir, output_filename)


censos.to_csv(output_path, index=False, encoding='utf-8-sig')